# NLP Exercise 3: Fair RAG
---


## Download packages and import modules

In [ ]:
%pip install langchain
%pip install langchain_text_splitters
%pip install langchain-community
%pip install langchain-chroma
%pip install -U langchain-ollama

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.



   ---------------------------------------- 2/2 [langchain-ollama]

Note: you may need to restart the kernel to use updated packages.


In [1]:
from typing import Literal
import numpy as np
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms.ollama import Ollama
import gradio as gr

## Text preprocessing

In [2]:
# Define chroma path
chroma_path = 'chroma'
chroma_collection_name = 'aesop-fables'
text_filename = 'pg11339.txt'

In [3]:
# Load documents
with open(text_filename, 'r', encoding='utf-8') as file:
    story_text = file.read()

# Get only the story titles and content
chapter_start = "THE FOX AND THE GRAPES\n\n\nA hungry Fox"
chapter_end = "ILLUSTRATIONS\n\n\n[Illustration: THE HARE AND THE TORTOISE]"
start_idx = story_text.find(chapter_start)
end_idx = story_text.find(chapter_end)
text = story_text[start_idx:end_idx].strip()

## Prepare document list

In [4]:
fables = text.split('\n' * 5)
fable_docs: list[Document] = []
for i, fable in enumerate(fables):
    fable_title = fables[i].split('\n' * 3, 1)[0] # Get title at the beginning of each fable
    fable_title = '-'.join(fable_title.split()) # Replace whitespaces with '-'
    fable_title = ''.join(fable_title.split(',')) # Remove ','
    fable_id = f'{i}:{fable_title}'
    doc = Document(
        page_content=fable,
        metadata={
            'id': fable_id,
            'source': text_filename,
        }
    )
    fable_docs.append(doc)
fable_docs[-10:]

[Document(metadata={'id': '274:THE-DOG-CHASING-A-WOLF', 'source': 'pg11339.txt'}, page_content='THE DOG CHASING A WOLF\n\n\nA Dog was chasing a Wolf, and as he ran he thought what a fine fellow\nhe was, and what strong legs he had, and how quickly they covered the\nground. "Now, there\'s this Wolf," he said to himself, "what a poor\ncreature he is: he\'s no match for me, and he knows it and so he runs\naway." But the Wolf looked round just then and said, "Don\'t you\nimagine I\'m running away from you, my friend: it\'s your master I\'m\nafraid of."'),
 Document(metadata={'id': '275:GRIEF-AND-HIS-DUE', 'source': 'pg11339.txt'}, page_content='GRIEF AND HIS DUE\n\n\nWhen Jupiter was assigning the various gods their privileges, it so\nhappened that Grief was not present with the rest: but when all had\nreceived their share, he too entered and claimed his due. Jupiter was\nat a loss to know what to do, for there was nothing left for him.\nHowever, at last he decided that to him should belon

## Vector Embedding

In [5]:
def embedding_function():
    embeddings = OllamaEmbeddings(model='nomic-embed-text')
    return embeddings

In [6]:
# Add documents to the Chroma DB using the `OllamaEmbeddings`
def add_to_chroma(documents: list[Document]):
    # Load the database
    db = Chroma(
        collection_name=chroma_collection_name,
        persist_directory=chroma_path,
        embedding_function=embedding_function()
    )

    # Add or update the documents
    existing_items = db.get(include=[])
    existing_ids = set(existing_items["ids"])
    print(f"Number of existing documents in DB: {len(existing_ids)}")

    new_docs = []
    for doc in documents:
        if doc.metadata["id"] not in existing_ids:
            new_docs.append(doc)

    if len(new_docs):
        print(f"Adding new documents: {len(new_docs)}")
        new_docs_ids = [doc.metadata["id"] for doc in new_docs]
        db.add_documents(new_docs, ids=new_docs_ids)
       # db.persist()
    else: 
        print("No new documents to add")

In [24]:
add_to_chroma(fable_docs)

Number of existing documents in DB: 0
Adding new documents: 284


## Query Data

In [ ]:
used_fables = [] # List of story content of used fables
random_doc = [None] # For randomised fairness option

In [22]:
FairnessOptions = Literal['sequential', 'randomised']
PROMPT_TEMPLATE = """
Answer the question based only on the following context:

{context}

---

Answer the question based on the above context: {question}
"""

def fable_query_rag(query_text: str, fairness: FairnessOptions, k = 10):
    """
    Generate text response from query text and context of relevant fables.

    Args:
        query_text (str): The query text from user.
        fairness (Literal['sequential', 'randomised']): Fairness option to be chosen.
            - Sequential fairness: Search for k, say 10 fables, and pick one for the answer and add it to the "used fables" list, which is to contain k-1 last fables used in the answer. After this, every time the pick top match that does not appear in the list, add it to the list, and drop the oldest entry from the list. 
            - Randomised fairness: Search for k, say 10, fables and randomly pick one. 
        k (int): Number of relevant fables to be retrieved.

    Returns:
        tuple:
            response_text (str): The response text generated by the Ollama model.\n
            sources (list[str]): List of fable document IDs.
    """
    # Prepare the DB.
    db = Chroma(
        collection_name=chroma_collection_name,
        persist_directory=chroma_path,
        embedding_function=embedding_function()
    )

    # Search the DB.
    results = db.similarity_search_with_score(query_text, k)

    # Combine the results into a single context string.
    if fairness == 'sequential':
        doc, _score = results[0] # Get top fable and its score
        used_fables_ids = [doc.metadata['id'] for doc in used_fables]
        if doc.metadata['id'] not in used_fables_ids:
            used_fables.append(doc) # Add fable if not in used_fables list
            if len(used_fables) > k - 1:
                used_fables.pop(0) # Drop the oldest entry from the list
        context_text = "\n\n---\n\n".join([doc.page_content for doc in used_fables])
    elif fairness == 'randomised':
        random_idx = np.random.randint(k - 1)
        doc, _score = results[random_idx]
        random_doc.pop()
        random_doc.append(doc)
        context_text = doc.page_content
    else:
        raise ValueError('\'fairness\' must be \'sequential\' or \'randomised\'')
    
    # Format the prompt with the context and the query.
    prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
    prompt = prompt_template.format(context=context_text, question=query_text)

    # Invoke the model with the formatted prompt.
    model = Ollama(model="mistral")
    response_text = model.invoke(prompt)

    # Extract the sources from the results.
    sources = [doc.metadata.get("id", None) for doc, _score in results]
    return response_text, sources

In [17]:
# Set fairness option and number of latest used fables here
fairness_option: FairnessOptions = 'sequential'
k = 10

In [23]:
# Gradio chat function
def fable_gradio_chat(message, history):
    answer, sources = fable_query_rag(message, fairness_option, k)
    sources_txt = ', '.join(sources)
    response_text = f'''{answer}
    
    -----------------------------------------
    Sources (after searching from Chroma DB): {sources_txt}

    '''
    if fairness_option == 'sequential':
        used_fables_txt = ', '.join([fable.metadata['id'] for fable in used_fables])
        response_text += f'Used fables: {used_fables_txt}'
    elif fairness_option == 'randomised':
        response_text += f'The random chosen fable (from Chroma DB search result): {random_doc[0].metadata['id']}'
    return response_text

In [ ]:
used_fables.clear() # Clear previous used fables before relaunching the chatbot
gr.ChatInterface(
    fable_gradio_chat,
    title='Fable Chat',
    examples=['Give me an example fable that contains both human and animal characters.', 'Why did the man and the wife kill the goose?', 'Did Jackdaw escape?'],
).launch(theme='soft')

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
